# Lecture audio with Chatterbox (free GPU: Kaggle or Google Colab)

Makes the spoken lectures of Free Exercise Book ahead of time with the open Chatterbox voice
model (MIT licence). See `tools/lecture-audio/README.md` in the repository.

**Kaggle:** Settings → verify your phone number first (GPUs stay hidden until then). Then in the
right-hand panel: Accelerator **GPU T4 x2**, Internet **on**. Output is kept with the notebook.

**Google Colab:** Runtime → Change runtime type → **T4 GPU**. Clips are saved to your Google
Drive (`MyDrive/lecture-audio/clips`), so a disconnected session just continues when run again.
Keep the browser tab open while it runs. If Colab asks to restart the session after
the install cell, restart and run all cells again.

1. First run with `SAMPLE = True` (a few minutes) and listen to the clips at the end.
2. Then set `SAMPLE = False` and run again (on Kaggle: **Save Version → Save & Run All**).
3. Finished clips are skipped, so run again until everything is made.
4. Download the clips folder and run `tools/lecture-audio/import.py` on your computer.

In [ ]:
SAMPLE = True          # a few clips from one topic, to listen to first
TOPICS = ""            # e.g. "math.integral.area-length-volume" (empty: every lecture)
EXAGGERATION = 0.45    # expressiveness, 0.25 (flat) to 1.0 (dramatic)
CFG = 0.5              # lower is slower and calmer
MAX_MINUTES = 650      # stop cleanly before Kaggle's 12-hour limit

In [ ]:
!pip install -q chatterbox-tts==0.1.7 "setuptools<81"
!rm -rf repo && git clone -q --depth 1 https://github.com/Kroy665/free_exercise_book.git repo
!nvidia-smi -L; ffmpeg -version | head -1

In [ ]:
# Where clips go. Kaggle: the notebook's output (add an earlier output as input to resume).
# Colab: Google Drive, so work survives a disconnected session.
import glob, os, shutil
COLAB = "google.colab" in str(get_ipython())
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/lecture-audio/clips"
else:
    OUT = "/kaggle/working/clips"
os.makedirs(OUT, exist_ok=True)
for d in glob.glob("/kaggle/input/*/clips"):
    shutil.copytree(d, OUT, dirs_exist_ok=True)
LOGS = os.path.dirname(OUT)
print(sum(len(f) for _, _, f in os.walk(OUT)), "clips already made in", OUT)

In [ ]:
# One process per GPU, each making every other clip.
import subprocess, torch, time
gpus = max(torch.cuda.device_count(), 1)
args = ["python3", "repo/tools/lecture-audio/generate.py", "repo/content", OUT,
        "--exaggeration", str(EXAGGERATION), "--cfg", str(CFG), "--max-minutes", str(MAX_MINUTES)]
if TOPICS: args += ["--topics", TOPICS]
if SAMPLE: args += ["--topics", TOPICS or "math.integral.area-length-volume", "--limit", "2"]
procs = []
for g in range(gpus):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    log = open(f"{LOGS}/gpu{g}.log", "w")
    procs.append(subprocess.Popen(args + ["--shard", f"{g}/{gpus}"], env=env, stdout=log, stderr=subprocess.STDOUT))
while any(p.poll() is None for p in procs):
    time.sleep(60)
    for g in range(gpus):
        lines = open(f"{LOGS}/gpu{g}.log").read().strip().splitlines()
        print(f"gpu{g}:", lines[-1] if lines else "starting")
for g in range(gpus):
    print(open(f"{LOGS}/gpu{g}.log").read()[-1500:])
print("exit codes:", [p.returncode for p in procs])

In [ ]:
# Listen to a few clips.
from IPython.display import Audio, display
for f in sorted(glob.glob(OUT + "/*/*.mp3"))[:4]:
    print(f); display(Audio(f))